# Demo Modul 6: CNN Lanjutan dan Transfer Learning

**Durasi sesi:** 120 menit  
**Kasus:** dataset citra kecil pada `data/raw/transfer/`, satu subfolder per kelas.

Modul 5 melatih CNN dari nol pada sepuluh ribu citra. Notebook ini menghadapi keadaan yang jauh lebih lazim: data berlabel hanya beberapa ratus citra.

## Capaian demo

Setelah demo, praktikan dapat:

1. memuat model pralatih beserta transformasi masukan yang menyertainya;
2. mengganti head dan menghitung parameter terlatih tiap strategi;
3. **membuktikan** bahwa pembekuan benar-benar terjadi;
4. membandingkan empat strategi pada anggaran yang sama; dan
5. menilai dampak augmentasi serta membaca kesalahan klasifikasi.

In [ ]:
import copy
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE)})

## 1. Bobot pralatih membawa aturan praprosesnya sendiri

Kesalahan paling merusak pada modul ini bukan arsitektur, melainkan normalisasi yang tidak cocok dengan bobot pralatih. Karena itu transformasi diambil **dari objek bobotnya**, bukan ditulis ulang.

In [ ]:
bobot = ResNet18_Weights.IMAGENET1K_V1
tf_eval = bobot.transforms()
print(tf_eval)

MEAN, STD = tf_eval.mean, tf_eval.std
SISI = tf_eval.crop_size[0]
print(f'\nukuran crop : {SISI}')
print(f'mean        : {MEAN}')
print(f'std         : {STD}')

# Augmentasi WAJIB memakai normalisasi yang sama.
tf_aug = T.Compose([
    T.RandomResizedCrop(SISI, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

## 2. Dataset dan split terstratifikasi

Dua objek `ImageFolder` dibuat atas folder yang sama: satu dengan augmentasi untuk latih, satu tanpa augmentasi untuk validasi dan uji. Indeksnya dibagi terstratifikasi.

In [ ]:
ROOT = Path('../../data/raw/transfer')
if not ROOT.exists():
    ROOT = Path('data/raw/transfer')

ds_aug = ImageFolder(ROOT, transform=tf_aug)      # dipakai split latih
ds_polos = ImageFolder(ROOT, transform=tf_eval)   # dipakai validasi dan uji
KELAS = ds_polos.classes
y = np.array(ds_polos.targets)

idx_latih, idx_sisa = train_test_split(
    np.arange(len(y)), train_size=0.70, stratify=y, random_state=SEED)
idx_val, idx_uji = train_test_split(
    idx_sisa, train_size=0.50, stratify=y[idx_sisa], random_state=SEED)

ds_latih = Subset(ds_aug, idx_latih)
ds_val = Subset(ds_polos, idx_val)
ds_uji = Subset(ds_polos, idx_uji)

print(f'kelas ({len(KELAS)}): {KELAS}')
print(f'latih {len(ds_latih)}  validasi {len(ds_val)}  uji {len(ds_uji)}')
print('citra per kelas (seluruh data):', np.bincount(y).tolist())

**Pemeriksaan:** augmentasi hanya melekat pada `ds_aug`. Split validasi dan uji mengambil dari `ds_polos`, sehingga keduanya bebas dari augmentasi apa pun.

## 3. Mengganti head dan menghitung parameter terlatih

In [ ]:
K = len(KELAS)

def buat_model(pralatih=True):
    seed_everything(SEED)
    m = resnet18(weights=bobot if pralatih else None)
    m.fc = nn.Linear(m.fc.in_features, K)          # head baru
    return m.to(DEVICE)

model = buat_model()
total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.fc.parameters())
layer4 = sum(p.numel() for p in model.layer4.parameters())

print(f'parameter total            : {total:,}')
print(f'head Linear(512,{K})         : {head:,}  ({100*head/total:.3f}%)')
print(f'layer4 + head              : {layer4 + head:,}  ({100*(layer4+head)/total:.1f}%)')
print(f'full fine-tuning           : {total:,}  (100%)')

**Temuan yang layak direnungkan:** membuka `layer4` saja sudah membuka sekitar **tiga perempat** parameter ResNet18. Penghematan partial fine-tuning karena itu bukan terutama pada jumlah parameter, melainkan pada komputasi mundur yang berhenti di batas blok beku.

## 4. Membekukan — dan membuktikannya

`requires_grad=False` belum cukup sebagai bukti. Parameter beku yang tetap masuk optimizer masih dapat bergerak akibat weight decay atau momentum. Karena itu kita periksa **dua** hal.

In [ ]:
def bekukan(model, strategi):
    for p in model.parameters():
        p.requires_grad = False
    if strategi == 'frozen':
        untuk_dilatih = [model.fc]
    elif strategi == 'partial':
        untuk_dilatih = [model.layer4, model.fc]
    else:                                   # 'full' atau 'scratch'
        untuk_dilatih = [model]
    for m in untuk_dilatih:
        for p in m.parameters():
            p.requires_grad = True
    return model

def n_terlatih(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for strategi in ('frozen', 'partial', 'full'):
    m = bekukan(buat_model(), strategi)
    print(f'{strategi:>8}: {n_terlatih(m):>11,} parameter terlatih '
          f'({100*n_terlatih(m)/total:5.1f}%)')

In [ ]:
# Bukti kedua: bobot beku benar-benar tidak bergerak setelah satu update.
m = bekukan(buat_model(), 'frozen')
salinan = m.layer1[0].conv1.weight.detach().clone()

opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
xb, yb = next(iter(DataLoader(ds_latih, batch_size=8, shuffle=False)))
opt.zero_grad()
nn.CrossEntropyLoss()(m(xb.to(DEVICE)), yb.to(DEVICE)).backward()
opt.step()

selisih = (m.layer1[0].conv1.weight.detach() - salinan).abs().max().item()
print(f'selisih maksimum bobot layer1 setelah satu update: {selisih:.2e}')
assert selisih == 0.0, 'bobot beku ternyata berubah'
print('pembekuan terbukti')

Perhatikan baris `torch.optim.Adam([p for p in m.parameters() if p.requires_grad], ...)`. Mengirim seluruh parameter ke optimizer adalah penyebab paling sering bobot beku ikut bergerak.

## 5. Empat run terkendali

Data, seed, batch size, dan jumlah epoch sama untuk keempatnya. Yang berbeda hanya bobot awal dan bagian mana yang boleh berubah.

In [ ]:
BATCH, EPOCH = 32, 5

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds):
    model.eval()
    kriteria = nn.CrossEntropyLoss(reduction='sum')
    total_loss, benar = 0.0, 0
    for xb, yb in loader(ds, 64, False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += kriteria(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(ds), benar / len(ds)

def jalankan(strategi, lr, label, epoch=EPOCH):
    model = bekukan(buat_model(pralatih=(strategi != 'scratch')), strategi)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    kriteria = nn.CrossEntropyLoss()
    dl = loader(ds_latih, BATCH, True)
    riwayat, mulai = {'val_acc': [], 'val_loss': []}, time.perf_counter()

    for _ in range(epoch):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            kriteria(model(xb), yb).backward()
            opt.step()
        vl, va = evaluasi(model, ds_val)
        riwayat['val_loss'].append(vl); riwayat['val_acc'].append(va)

    tl, _ = evaluasi(model, ds_latih)
    durasi = time.perf_counter() - mulai
    return model, riwayat, {
        'run_id': label, 'seed': SEED, 'strategi': strategi,
        'bobot_awal': 'acak' if strategi == 'scratch' else 'pralatih',
        'learning_rate': lr, 'parameter_total': total,
        'parameter_terlatih': n_terlatih(model), 'augmentasi': 'ringan',
        'train_loss': tl, 'val_loss': riwayat['val_loss'][-1],
        'val_acc': riwayat['val_acc'][-1], 'gap': riwayat['val_loss'][-1] - tl,
        'detik_per_epoch': durasi / epoch,
    }

hasil, kurva, model_simpan = [], {}, {}
for strategi, lr, label in [('scratch', 1e-3, 'from-scratch'),
                            ('frozen', 1e-3, 'frozen-backbone'),
                            ('partial', 1e-4, 'partial-layer4'),
                            ('full', 1e-4, 'full-finetune')]:
    m, r, catatan = jalankan(strategi, lr, label)
    hasil.append(catatan); kurva[label] = r; model_simpan[label] = m

pd.DataFrame(hasil)[['run_id', 'bobot_awal', 'parameter_terlatih', 'val_loss',
                     'val_acc', 'gap', 'detik_per_epoch']]

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for label, r in kurva.items():
    ax.plot(range(1, EPOCH + 1), r['val_acc'], marker='o', label=label)
ax.set_xlabel('epoch'); ax.set_ylabel('validation accuracy')
ax.set_title('Empat strategi pada anggaran yang sama'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

Bacalah tiga kolom bersama-sama: `parameter_terlatih`, `val_acc`, dan `detik_per_epoch`. Frozen backbone melatih sekitar $0{,}02\%$ parameter — dan pada data kecil sering sudah mengalahkan model yang dilatih dari nol.

## 6. Augmentasi: dua tingkat pada strategi terbaik

In [ ]:
juara = pd.DataFrame(hasil).sort_values('val_loss').iloc[0]
print('strategi terbaik:', juara['run_id'])

tf_keras = T.Compose([
    T.RandomResizedCrop(SISI, scale=(0.3, 1.0)),
    T.RandomRotation(45),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

peta_lr = {'scratch': 1e-3, 'frozen': 1e-3, 'partial': 1e-4, 'full': 1e-4}
for nama_aug, tf in [('tanpa', tf_eval), ('agresif', tf_keras)]:
    ds_aug.transform = tf
    _, _, catatan = jalankan(juara['strategi'], peta_lr[juara['strategi']],
                             f"aug-{nama_aug}")
    catatan['augmentasi'] = nama_aug
    hasil.append(catatan)
ds_aug.transform = tf_aug          # kembalikan ke augmentasi ringan

tabel = pd.DataFrame(hasil)
tabel[['run_id', 'strategi', 'augmentasi', 'val_loss', 'val_acc', 'gap']]

Augmentasi agresif dapat menurunkan akurasi karena rotasi dan pemotongan besar bisa menghapus ciri yang justru menentukan kelasnya. Augmentasi bukan selalu menolong — ia harus dipilih sesuai jenis citranya.

## 7. Evaluasi test satu kali dan analisis kesalahan

In [ ]:
final = model_simpan[juara['run_id']]
test_loss, test_acc = evaluasi(final, ds_uji)
print(f"validation acc : {juara['val_acc']:.4f}")
print(f"test acc       : {test_acc:.4f}   (test dipakai SATU kali)")

@torch.no_grad()
def prediksi(model, ds):
    model.eval()
    p, t = [], []
    for xb, yb in loader(ds, 64, False):
        p.append(model(xb.to(DEVICE)).argmax(1).cpu()); t.append(yb)
    return torch.cat(p), torch.cat(t)

pred, target = prediksi(final, ds_uji)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(target, pred, display_labels=KELAS,
                                        xticks_rotation=45, colorbar=False, ax=ax)
plt.tight_layout(); plt.show()

salah = (pred != target).nonzero().flatten()[:5]
for i in salah:
    print(f'  indeks {i.item():>4}: benar={KELAS[target[i]]:>14}  '
          f'diprediksi={KELAS[pred[i]]}')

## Exit ticket

1. Mengapa transformasi masukan harus diambil dari objek bobot pralatih?
2. Dua bukti apa yang diperlukan sebelum menyatakan sebuah layer benar-benar beku?
3. Frozen backbone melatih sekitar $0{,}02\%$ parameter — mengapa hasilnya bisa mengalahkan model yang dilatih dari nol?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb` — empat run strategi, dua run augmentasi, satu evaluasi test, dan rekomendasi yang menyebut akurasi, waktu, serta parameter terlatih sekaligus.